In [ ]:
# pip install FlagEmbedding

In [6]:
import os
from typing import Any

from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import re
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder
import time
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import CrossEncoder

"""
RAG流程:
加载文件
文档切分
嵌入模型
存入向量数据库
混合检索(语义检索 BM25检索 RRF融合)
重排序(CROSS-ENCODER)
上下文组装

定义RAG_service类:
    配置文件
    配置数据库

    定义方法:
        添加文档
        检索文档

"""


class RAG_service:

    def __init__(
            self,
            index_name: str,
            api_key: str,
            cloud: str,
            region: str,
            dimension: int = 1024
    ):
        """
        创建初始化类
        包含:
        初始化Pinecone
        md文档分割工具
        递归分块(段落 句子)
        BGA向量化
        BM25稀疏矩阵向量化
        BAAI重排序模型

        :param index_name:
        :param api_key:
        :param cloud:
        :param region:
        :param dimension:
        """
        self.index_name = index_name
        self.api_key = api_key
        self.cloud = cloud
        self.region = region
        self.dimension = dimension
        self.pc = Pinecone(api_key=api_key)
        self.index = self.pc.Index(self.index_name)

        # 初始化工具
        headers_to_split_on = [("#", "Header_1")]

        self.md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=512,
            chunk_overlap=50,
            separators=["\n\n", "\n", "。", "；", " "],
            add_start_index=True
        )

        self.reranker = CrossEncoder('BAAI/bge-reranker-large',  max_length=512)

        # 稀疏向量
        self.bm25 = BM25Encoder().load("bm25_law_params.json")

        # 密集向量
        model_name = "BAAI/bge-large-zh-v1.5"
        self.embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            encode_kwargs={'normalize_embeddings': True}
        )

    def create_index(self, wait_for_completion: bool = True) -> bool:
        """
        创建索引
        创建混合索引
        指定的索引方式
        :param wait_for_completion:
        :return:
        """
        self.pc = Pinecone(api_key=self.api_key)

        # 混合索引的强制要求：metric 必须为 dotproduct，vector_type 为 dense
        target_metric = "dotproduct"

        # 检查索引是否存在
        if not self.pc.has_index(self.index_name):
            print(f"正在创建混合检索索引: {self.index_name}...")
            self.pc.create_index(
                name=self.index_name,
                dimension=self.dimension,
                metric=target_metric,
                spec=ServerlessSpec(cloud=self.cloud, region=self.region),
                vector_type="dense",
            )
        else:
            print(f"索引 '{self.index_name}' 已存在。")
        # 等待索引就绪 异步处理
        if wait_for_completion:
            while not self.pc.describe_index(self.index_name).status.get("ready", False):
                time.sleep(2)
            print(f"索引 '{self.index_name}' 已就绪。")

        self.index = self.pc.Index(self.index_name)
        print(f"索引 '{self.index_name}' 已就绪。")

        return True

    def get_index_stats(self):
        """
        获取索引是否创建成功
        测试索引是否创建成功并打印统计信息
        :return: 是否创建成功
        """
        stats = self.index.describe_index_stats()
        print("当前索引的状态: ", stats)
        return True

    def get_Documents(self, file_path: str) -> list[Any] | None:
        """
        添加文本到数据库中需要:
        加载文本,
        文本分块,

        具体实施:
        (正则表达式)
        将清洗好的文件加载
        以每一个案例为单位
        先提取元数据
        (分块)
        从[基本案情]到最后的内容提取
        以句子为单位,合成一大段

        :param file_path:str
        :return 符合输入格式的列表
        """

        # 加载获取
        global articles, metadata, facts_cleaned
        loader = TextLoader(
            file_path,
            encoding="utf-8"
        )

        # 加载一次之后可以反复使用
        # 稀疏向量
        bm25 = BM25Encoder().load("lawApp_LangGraph/RAG_service/bm25_law_params.json")

        # 密集向量
        model_name = "BAAI/bge-large-zh-v1.5"
        embeddings = HuggingFaceEmbeddings(model_name=model_name, encode_kwargs={'normalize_embeddings': True})

        # 默认读取的为Document形式
        documents = loader.load()

        for doc in documents:
            # split_text 返回 List[Document]，每个 Document 对应一个一级标题下的内容块
            articles = self.md_splitter.split_text(doc.page_content)

        # 获取元数据和正文内容
        for i, article in enumerate(articles):

            metadata = {}

            # 提取年份：假设文件名或路径包含 202X
            year_match = re.search(r"20\d{2}", article.page_content)
            if year_match:
                metadata["year"] = year_match.group(0) if year_match else "Unknown"

                # 提取裁判书字号：匹配如（2023）最高法民终...号
                case_num_pattern = r'裁判书字号[\s\\n]+((?:(?!裁判书字号)[\s\S])+?法院[\s\S]+?书)'
                case_num_match = re.search(case_num_pattern, article.page_content)

                metadata["case_number"] = case_num_match.group(1).strip() if case_num_match else "未识别"

                # 提取案由：通常在字号之后，或者是特定的段落
                case_cause_pattern = r"案由[:：]\s*([\u4e00-\u9fa5]+)"
                cause_match = re.search(case_cause_pattern, article.page_content)

                metadata["case_cause"] = cause_match.group(1) if cause_match else "通用"

                # 提取基本案情
                facts_pattern = r'【基本案情】\s*([\s\S]+?)(?=\n【|$)'
                facts_match = re.search(facts_pattern, article.page_content)

                # 获取捕获组内容（不含【基本案情】）
                raw_content = facts_match.group(1)
                # 去除空格、换行、制表符等所有空白字符，以及 # 符号
                facts_cleaned = re.sub(r'\n+', '\n', raw_content).strip()  # 去除所有空白（空格、换行等）
                facts_cleaned = facts_cleaned.replace('#', '')  # 去除所有 # 字符

        # 插入Pinecone数据容器
        Pinecone_records = []

        chunks = self.text_splitter.split_text(facts_cleaned)

        for i, chunk in enumerate(chunks):
            record_id = f"doc_chunk_{i}"

            # 密集向量和稀疏向量
            dense_vector = embeddings.embed_query(chunk)

            # 返回 {"indices": [...], "values": [...]}
            sparse_vector = bm25.encode_documents(chunk)

            """
            添加内容: id 向量数据 元数据:{年份 判决书 案由 文档切片}
            符合Pinecone的输入格式
            """
            record = {
                "id": record_id,
                "chunk_index": i,
                "values": dense_vector,  # 稠密向量列表 [0.12, -0.34, ...]
                "sparse_values": sparse_vector,  # 稀疏向量字典
                "metadata": {
                    **metadata,
                    "chunk_text": chunk,
                }
            }

            Pinecone_records.append(record)

            return Pinecone_records

    def add_document(self, file_path: str, namespace: str, ):
        """
        添加分块好的文本到数据库中
        :param file_path:
        :param namespace:
        :return:
        """
        records = self.get_Documents(file_path=file_path)

        # 指定命名空间添加数据
        self.index.upsert(
            vectors=records,
            namespace=namespace,
        )

        return True

    def search_withDenseSparse(
            self,
            query: str,
            namespace: str,
            top_k: int = 50,
            rerank_top_n=10,
            alpha: float = 0.5
    ) -> list:
        """
        分别获取问题的稀疏和密集向量化矩阵
        双路查询
        对结果和文字重排序
        分别检索的向量对文本意思没有关联
        交叉编码器同时接收查询‑文档对作为输入，通过 Transformer 的全注意力机制（Self‑Attention）让查询和文档的每个词充分交互，最终输出一个相关性分数

        :param query:
        :param namespace:
        :param top_k:
        :param rerank_top_n:
        :param alpha:
        :return: 重排序后的列表
        """
        # 密集向量
        dense_vec = self.embeddings.embed_query(query)

        # 稀疏向量
        sparse_vec = self.bm25.encode_documents(query)

        # 混合召回
        results = self.index.query(
            vector=dense_vec,
            sparse_vector=sparse_vec,
            alpha=alpha,        # 控制权重：关键词 (0 <====> 1) 语义
            namespace=namespace,
            top_k=top_k,
            include_metadata=True
        )
        print("混合召回中...")

        #  提取文本，准备重排序
        matches = results.matches
        texts = [m.metadata['chunk_text'] for m in matches]
        # 问题与原文的键值对
        pairs = [[query, t] for t in texts]

        # 3. 计算重排序分数
        scores = self.reranker.predict(pairs)

        # 4. 重组并排序
        for match, score in zip(matches, scores):
            match.rerank_score = score

        # 按分数高低排序
        reranked = sorted(matches, key=lambda x: x.rerank_score, reverse=True)

        print("重排序中...")

        print("重排序结果为:{}".format(reranked))

        return reranked[:rerank_top_n]








In [3]:
from sentence_transformers import CrossEncoder
model = CrossEncoder('BAAI/bge-reranker-large')
score = model.predict([("查询文本", "文档文本")])
print(score)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[0.5837171]


In [7]:

load_dotenv()

# 实例化测试
service = RAG_service(
    index_name="pinecone-test-lawapp",
    api_key=os.getenv("PINECONE_API_KEY"),
    cloud="aws",
    region="us-east-1",
)

# service.get_index_stats()

service.search_withDenseSparse("离婚抚养权", "Law_test_namespace", 5,10,0.6)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


混合召回中...
重排序中...
重排序结果为:[{'id': 'Docu25_chunk6',
 'metadata': {'case_cause': '同居关系子女抚养纠纷',
              'case_number': '福建省福清市人民法院(2017)闽0181民初第4447号民事判决书',
              'chunk_index': 6,
              'chunk_text': '根据《中华人民共和国婚姻法 '
                            '》第二十一条之规定，父母对子女有抚养教育的义务；子女对父母有赡养扶助的义务。父母不履行抚养义务时，未成年的或不能独立生活的子女，有要求父母付给抚养费的权利。子女不履行赡养义务时，无劳动能力的或生活困难的父母，有要求子女付给赡养费的权利。《最高人民法院关于人民法院审理离婚案件处理子女抚养问题的若干具体意见》第十一条规定，抚育费的给付期限，一般至子女十八周岁为止。十六周岁以上不满十八周岁，以其劳动收入为主要生活来源，并能维持当地一般生活水平的，父母可停止给付抚育费。《最高人民法院关于适用〈中华人民共和国婚姻法〉若干问题的解释（一）》第二十一条规定的婚姻法第二十一条所称“抚养费”，包括子女生活费、教育费、医疗费等费用。抚养费的法律属性和特点有三：(1)抚养义务的无条件性。父母对未成年子女的抚养义务是无条件的，子女一旦出生，无论父母经济条件及负担能力如何，也不论是否愿意，均必须依法承担抚养义务。即使降低自己的生活水平、牺牲自己的事业发展和生活享受，也必须首先保障子女的生存和生活。(2)抚养内容的复合性',
              'year': '2017'},
 'rerank_score': 0.9850637,
 'score': 0.676398277,
 'values': []}, {'id': 'Docu33_chunk2',
 'metadata': {'case_cause': '变更抚养关系纠纷',
              'case_number': '福建省泉州市中级人民法院(2017)闽05民终第604号民事判决书',
              'chunk_index': 2,
              'chunk

[{'id': 'Docu25_chunk6',
  'metadata': {'case_cause': '同居关系子女抚养纠纷',
               'case_number': '福建省福清市人民法院(2017)闽0181民初第4447号民事判决书',
               'chunk_index': 6,
               'chunk_text': '根据《中华人民共和国婚姻法 '
                             '》第二十一条之规定，父母对子女有抚养教育的义务；子女对父母有赡养扶助的义务。父母不履行抚养义务时，未成年的或不能独立生活的子女，有要求父母付给抚养费的权利。子女不履行赡养义务时，无劳动能力的或生活困难的父母，有要求子女付给赡养费的权利。《最高人民法院关于人民法院审理离婚案件处理子女抚养问题的若干具体意见》第十一条规定，抚育费的给付期限，一般至子女十八周岁为止。十六周岁以上不满十八周岁，以其劳动收入为主要生活来源，并能维持当地一般生活水平的，父母可停止给付抚育费。《最高人民法院关于适用〈中华人民共和国婚姻法〉若干问题的解释（一）》第二十一条规定的婚姻法第二十一条所称“抚养费”，包括子女生活费、教育费、医疗费等费用。抚养费的法律属性和特点有三：(1)抚养义务的无条件性。父母对未成年子女的抚养义务是无条件的，子女一旦出生，无论父母经济条件及负担能力如何，也不论是否愿意，均必须依法承担抚养义务。即使降低自己的生活水平、牺牲自己的事业发展和生活享受，也必须首先保障子女的生存和生活。(2)抚养内容的复合性',
               'year': '2017'},
  'rerank_score': 0.9850637,
  'score': 0.676398277,
  'values': []},
 {'id': 'Docu33_chunk2',
  'metadata': {'case_cause': '变更抚养关系纠纷',
               'case_number': '福建省泉州市中级人民法院(2017)闽05民终第604号民事判决书',
               'chunk_index': 2,
               'chunk_text': '【